In [ ]:
# 1. IMPORTATION DES LIBRAIRIES
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder, StandardScaler, QuantileTransformer


In [ ]:
# 1. CHARGEMENT ET CONCATÉNATION

# Chargement des datasets d'entraînement et de test
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

# Outliers à enlever selon la documentation
train_df = train_df[train_df["GrLivArea"] < 4000].reset_index(drop=True)

ntrain = len(train_df)
y_train_log = np.log1p(train_df["SalePrice"].copy())

all_data = pd.concat([train_df.drop(columns=["SalePrice"]), test_df], axis=0).reset_index(drop=True)

# Sauvegarde des IDs pour la soumission finale Kaggle, puis suppression
test_ids = test_df['Id'].copy()
all_data = all_data.drop(['Id'], axis=1)

print(f"Shape initiale all_data : {all_data.shape}")


In [ ]:
missing_counts = all_data.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

print(f"\nDONNÉES MANQUANTES ({len(missing_counts)} colonnes):")
if len(missing_counts) > 0:
    for col, count in missing_counts.head(100).items():
        pct = 100 * count / len(all_data)
        print(f"   {col:20s} : {count:4d} ({pct:5.1f}%)")
else:
    print("   Aucune")

In [ ]:
# Suppression des lignes contenant des NA considérés comme des erreurs dans train.csv
# Renseigne ici UNIQUEMENT les colonnes où un NA est une vraie erreur de saisie.
error_missing_cols = [
    'MSZoning',
    'BsmtFullBath', 
    'Functional',
    'BsmtHalfBath',
    'Utilities',
    'BsmtFinSF1',
    'Exterior2nd',
    'Exterior1st',
    'Electrical',
    'TotalBsmtSF',
    'BsmtUnfSF',
    'BsmtFinSF2',
    'KitchenQual',
    'GarageArea',
    'GarageCars',
    'SaleType'
]

print(f"Colonnes à vérifier pour NA erreurs : {error_missing_cols}")

if len(error_missing_cols) > 0:
    valid_error_cols = [c for c in error_missing_cols if c in train_df.columns]
    dropped_error_cols = [c for c in error_missing_cols if c not in train_df.columns]

    if len(dropped_error_cols) > 0:
        print(f"Colonnes ignorées (absentes de train.csv): {dropped_error_cols}")

    n_before_drop = len(train_df)
    train_df = train_df.dropna(subset=valid_error_cols).reset_index(drop=True)
    n_removed = n_before_drop - len(train_df)
    print(f"Lignes supprimées sur train_df (NA erreurs): {n_removed}")
else:
    print("Aucune suppression de lignes: error_missing_cols est vide.")

In [ ]:
# 2. NETTOYAGE GLOBAL SUR ALL_DATA

def apply_base_preprocessing(data):
    """Applique le nettoyage de base sur l'ensemble complet (Train + Test)"""
    df = data.copy()

    # 1. NA signifiant "Absence de l'équipement" -> Catégorie 'None'
    cols_none = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
                 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']
    for col in cols_none:
        if col in df.columns:
            df[col] = df[col].fillna('None')

    # 2. NA signifiant 0 pour les variables numériques
    cols_zero = ['GarageYrBlt', 'MasVnrArea']
    for col in cols_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 3. Imputation par la médiane globale pour la façade (LotFrontage)
    if 'LotFrontage' in df.columns:
        df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

    # 5. Traitement Multicolinéarité
    cols_to_drop = ['GarageArea', 'TotRmsAbvGrd', 'GarageYrBlt']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # 6. Feature Engineering
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['AgeAtSale'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']

    return df

# Application du nettoyage
all_data_clean = apply_base_preprocessing(all_data)

# 3. SÉPARATION TRAIN / TEST

# On utilise ntrain pour retrouver nos données d'entraînement exactes
X_train_clean = all_data_clean.iloc[:ntrain].copy()
X_test_clean = all_data_clean.iloc[ntrain:].copy()

print(f"Shape X_train_clean : {X_train_clean.shape}")
print(f"Shape X_test_clean  : {X_test_clean.shape}")

In [ ]:
# 4. ENCODAGE AVANCÉ

class AdvancedCategoricalEngineer(BaseEstimator, TransformerMixin):
    """Transformateur Scikit-Learn pour encodage ordinal et Target Encoding (sklearn)."""
    def __init__(self, smoothing=10.0):
        self.smoothing = smoothing
        self.qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
        self.ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                             'HeatingQC', 'KitchenQual', 'FireplaceQu',
                             'GarageQual', 'GarageCond', 'PoolQC']
        self.te = None
        self.nominal_cols = None

    def fit(self, X, y):
        X_copy = X.copy()
        for col in self.ordinal_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.qual_mapping).fillna(0)

        self.nominal_cols = X_copy.select_dtypes(include=['object', 'string']).columns.tolist()

        for col in self.nominal_cols:
            X_copy[col] = X_copy[col].astype(str).fillna('Missing')

        self.te = TargetEncoder(
            categories='auto',
            target_type='continuous',
            smooth=self.smoothing,
            cv=5,
            shuffle=True,
            random_state=42
        )
        self.te.fit(X_copy[self.nominal_cols], y)
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.ordinal_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.qual_mapping).fillna(0)

        if self.nominal_cols:
            for col in self.nominal_cols:
                if col in X_copy.columns:
                    X_copy[col] = X_copy[col].astype(str).fillna('Missing')
            X_copy[self.nominal_cols] = self.te.transform(X_copy[self.nominal_cols])

        return X_copy


# 5. VALIDATION CROISÉE - RÉGRESSION LASSO

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

print("\nDébut Validation Croisée Lasso (5-Fold)...")
# Pipeline M1 version Lasso
pipeline_m1_lasso = Pipeline([
    ('encoder', AdvancedCategoricalEngineer(smoothing=10.0)),
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=0.0005, max_iter=20000))
])

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_clean)):

    # Séparation locale pour ce pli
    X_tr, y_tr = X_train_clean.iloc[train_idx], y_train_log.iloc[train_idx]
    X_val, y_val = X_train_clean.iloc[val_idx], y_train_log.iloc[val_idx]

    # Entraînement et Prédiction
    pipeline_m1_lasso.fit(X_tr, y_tr)
    preds = pipeline_m1_lasso.predict(X_val)

    fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_scores.append(fold_rmse)
    print(f"Fold {fold + 1} | RMSE: {fold_rmse:.5f}")

print(f"\nRMSE Moyen (CV) : {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})")

In [ ]:
# 4. ANALYSE MULTICOLINÉARITÉ
print("\n" + "="*70)
print("ANALYSE MULTICOLINÉARITÉ")
print("="*70)

numeric_features = X_train_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_clean.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
correlation_matrix = X_train_clean[numeric_features].corr()

print("\nCorrélations fortes (|r| > 0.85):")
print("-" * 60)

high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.85:
            col_i = correlation_matrix.columns[i]
            col_j = correlation_matrix.columns[j]
            corr_value = correlation_matrix.iloc[i, j]
            high_corr_pairs.append((col_i, col_j, corr_value))
            print(f"   {col_i:20s} <-> {col_j:20s} : {corr_value:.3f}")

if len(high_corr_pairs) == 0:
    print("   Aucune corrélation forte détectée")

# Heatmap
print("\nHeatmap de corrélation (variables numériques):")
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, annot=False, square=True, cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de corrélation - Variables numériques')
plt.tight_layout()
plt.show()

# PHASE 3 : MODÉLISATION (BASELINE → OPTIMISATION)

In [ ]:
# 9. PIPELINE OPTIMISÉ (QuantileTransformer + Date Transformer + Lasso avec alphas fins)
print("\n" + "="*70)
print("PIPELINE OPTIMISÉ (QuantileTransformer + Fine Alphas)")
print("="*70)

numeric_transformer_optimized = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', QuantileTransformer(output_distribution='normal', random_state=42))
])

date_transformer_optimized = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', QuantileTransformer(output_distribution='normal', random_state=42))
])

categorical_transformer_optimized = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_encoder', TargetEncoder(categories='auto', target_type='continuous', smooth='auto', cv=5))
])

# Valider que les colonnes listées existent bien (certaines peuvent avoir été supprimées précédemment)
numeric_features_valid = [c for c in numeric_features if c in X_train_clean.columns]
categorical_features_valid = [c for c in categorical_features if c in X_train_clean.columns]

missing_num = sorted(set(numeric_features) - set(numeric_features_valid))
missing_cat = sorted(set(categorical_features) - set(categorical_features_valid))
if missing_num or missing_cat:
    print("\nAttention: certaines colonnes listées comme features sont absentes après preprocessing:")
    if missing_num:
        print(f"   - Numériques manquantes ({len(missing_num)}): {missing_num}")
    if missing_cat:
        print(f"   - Catégoriques manquantes ({len(missing_cat)}): {missing_cat}")

preprocessor_optimized = ColumnTransformer(transformers=[
    ('num', numeric_transformer_optimized, numeric_features_valid),
    ('cat', categorical_transformer_optimized, categorical_features_valid)
])

lasso_optimized = LassoCV(alphas=np.logspace(-6, 1, 1000), cv=5, random_state=42, max_iter=10000, tol=1e-4)

pipeline_optimized = Pipeline(steps=[
    ('preprocessor', preprocessor_optimized),
    ('model', lasso_optimized)
])

print("\nPipeline optimisé créé:")
print("   • Numérique: Imputation (médiane) + QuantileTransformer")
print("   • Catégorique: Imputation (mode) + TargetEncoder(cv=5)")
print("   • Modèle: LassoCV(alphas=-6 à +1, 1000 points, cv=5)")

print("\n   Entraînement...")
# S'assurer que la cible alignée correspond bien au X utilisé (évite mismatch si train a été modifié)
y_train_for_fit = y_train_log.iloc[: X_train_clean.shape[0] ].reset_index(drop=True)
pipeline_optimized.fit(X_train_clean, y_train_for_fit)

y_pred_opt = pipeline_optimized.predict(X_train_clean)
rmse_opt = np.sqrt(mean_squared_error(y_train_for_fit, y_pred_opt))
mae_opt = mean_absolute_error(y_train_for_fit, y_pred_opt)
r2_opt = r2_score(y_train_for_fit, y_pred_opt)
alpha_opt = pipeline_optimized.named_steps['model'].alpha_

print(f"\nMÉTRIQUES OPTIMISÉES:")
print(f"   RMSE: {rmse_opt:.4f}")
print(f"   MAE: {mae_opt:.4f}")
print(f"   R²: {r2_opt:.4f}")
print(f"   Alpha: {alpha_opt:.6f}")

if rmse_opt <= 0.125:
    print(f"\n   OBJECTIF ATTEINT! RMSE {rmse_opt:.4f} ≤ 0.125")
else:
    print(f"\n   À {0.125 - rmse_opt:.4f} de l'objectif")

In [ ]:
# 9B. ENSEMBLE AVEC LASSO CONSERVÉ AU CŒUR DU PIPELINE
print("\n" + "="*70)
print("ENSEMBLE: LASSO + BOOSTING + RANDOM FOREST")
print("="*70)

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

ensemble_models = {
    "lasso": pipeline_optimized,
    "hgb": Pipeline(steps=[
        ('preprocessor', preprocessor_optimized),
        ('model', HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.05,
            max_iter=300,
            max_depth=6,
            random_state=42
        ))
    ]),
    "rf": Pipeline(steps=[
        ('preprocessor', preprocessor_optimized),
        ('model', RandomForestRegressor(
            n_estimators=400,
            max_depth=12,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

kf_ensemble = KFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = {}
test_predictions = {}
cv_rmse = {}

for name, model_template in ensemble_models.items():
    print(f"\nModèle: {name}")
    oof = np.zeros(X_train_clean.shape[0])
    fold_test_predictions = []

    for fold, (train_idx, val_idx) in enumerate(kf_ensemble.split(X_train_clean), 1):
        model_fold = clone(model_template)
        X_tr = X_train_clean.iloc[train_idx]
        X_val = X_train_clean.iloc[val_idx]
        y_tr = y_train_for_fit.iloc[train_idx]
        y_val = y_train_for_fit.iloc[val_idx]

        model_fold.fit(X_tr, y_tr)
        val_pred = model_fold.predict(X_val)
        test_pred = model_fold.predict(X_test_clean)

        oof[val_idx] = val_pred
        fold_test_predictions.append(test_pred)

        fold_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        print(f"   Fold {fold} | RMSE: {fold_rmse:.5f}")

    oof_predictions[name] = oof
    test_predictions[name] = np.mean(fold_test_predictions, axis=0)
    cv_rmse[name] = np.sqrt(mean_squared_error(y_train_for_fit, oof))
    print(f"   RMSE CV moyen: {cv_rmse[name]:.5f}")

weight_denominator = sum(1.0 / score for score in cv_rmse.values())
ensemble_weights = {name: (1.0 / score) / weight_denominator for name, score in cv_rmse.items()}

print("\nPoids du blend inverse-RMSE:")
for name, weight in ensemble_weights.items():
    print(f"   {name}: {weight:.3f}")

oof_ensemble = np.zeros(X_train_clean.shape[0])
y_pred_log_ensemble = np.zeros(X_test_clean.shape[0])
for name in ensemble_models:
    oof_ensemble += ensemble_weights[name] * oof_predictions[name]
    y_pred_log_ensemble += ensemble_weights[name] * test_predictions[name]

y_pred_dollars_ensemble = np.expm1(y_pred_log_ensemble)
rmse_ensemble = np.sqrt(mean_squared_error(y_train_for_fit, oof_ensemble))
mae_ensemble = mean_absolute_error(y_train_for_fit, oof_ensemble)
r2_ensemble = r2_score(y_train_for_fit, oof_ensemble)

print("\nMÉTRIQUES ENSEMBLE:")
print(f"   RMSE: {rmse_ensemble:.4f}")
print(f"   MAE: {mae_ensemble:.4f}")
print(f"   R²: {r2_ensemble:.4f}")

final_model_name = "Ensemble Lasso + HGB + RF"
final_rmse = rmse_ensemble
final_mae = mae_ensemble
final_r2 = r2_ensemble
final_y_pred_log = y_pred_log_ensemble
final_y_pred_dollars = y_pred_dollars_ensemble


# PHASE 5 : PRÉDICTIONS & SOUMISSION

In [ ]:
# 15. PRÉDICTIONS FINALES
print("\n" + "="*70)
print("PRÉDICTIONS FINALES")
print("="*70)

# Optimisé
y_pred_log_opt = pipeline_optimized.predict(X_test_clean)
y_pred_dollars_opt = np.expm1(y_pred_log_opt)

print(f"\nModèle Optimisé:")
print(f"   Min: ${y_pred_dollars_opt.min():,.0f}")
print(f"   Max: ${y_pred_dollars_opt.max():,.0f}")
print(f"   Médiane: ${np.median(y_pred_dollars_opt):,.0f}")
print(f"   Moyenne: ${y_pred_dollars_opt.mean():,.0f}")

if 'final_y_pred_dollars' in globals():
    print(f"\nModèle Ensemble:")
    print(f"   Min: ${final_y_pred_dollars.min():,.0f}")
    print(f"   Max: ${final_y_pred_dollars.max():,.0f}")
    print(f"   Médiane: ${np.median(final_y_pred_dollars):,.0f}")
    print(f"   Moyenne: ${final_y_pred_dollars.mean():,.0f}")
    print(f"   Sélection finale: {final_model_name}")
    y_pred_dollars_final = final_y_pred_dollars
else:
    y_pred_dollars_final = y_pred_dollars_opt
    final_model_name = "Modèle Optimisé Lasso"
    final_rmse = rmse_opt
    final_mae = mae_opt
    final_r2 = r2_opt

# Optuna predictions removed (Optuna pipeline was removed)
# If you want to reinstate Optuna, re-run tuning and training cells to recreate pipeline_optuna


In [ ]:
# 16. GÉNÉRATION FICHIERS CSV & RÉSUMÉ FINAL
print("\n" + "="*70)
print("GÉNÉRATION FICHIERS SUBMISSIONS")
print("="*70)

submission_opt = pd.DataFrame({
    'Id': test_ids.values,
    'SalePrice': y_pred_dollars_final
})
submission_opt.to_csv('M1_Lasso_V6_4.csv', index=False)
print(f"\nM1_Lasso_V6_4.csv généré ({len(submission_opt)} prédictions)")

# RÉSUMÉ FINAL (sans export Optuna)
print(f"\n{'='*70}")
print("RÉSUMÉ FINAL - SUBMISSION")
print(f"{'='*70}")

summary_text = f"""

MEILLEUR MODÈLE (parmi évalués):
    • Modèle: {final_model_name}
    • RMSE: {final_rmse:.4f}
    • MAE: {final_mae:.4f}
    • R²: {final_r2:.4f}
    • Scaler recommandé: QuantileTransformer

FICHIER GÉNÉRÉ:
    • M1_Lasso_V6_4.csv

"""

print(summary_text)
